In [13]:
import pandas as pd 


⏱️ Cell executed in: 0.001 sec


In [14]:
df=pd.read_csv("D:\ORBIT\COMMENT_ANALYZER\datasets\Sentiment.csv")


⏱️ Cell executed in: 4.256 sec


In [15]:
df.head()

,CommentID,VideoID,VideoTitle,AuthorName,AuthorChannelID,CommentText,Sentiment,Likes,Replies,PublishedAt,CountryCode,CategoryID
0,UgyRjrEdJIPrf68uND14AaABAg,mcY4M9gjtsI,They killed my friend.#tales #movie #shorts,@OneWhoWandered,UC_-UEXaBL1dqqUPGkDll49A,Anyone know what movie this is?,Neutral,0,2,2025-01-15 00:54:55,NZ,1
1,UgxXxEIySAwnMNw8D7N4AaABAg,2vuXcw9SZbA,Man Utd conceding first penalty at home in yea...,@chiefvon3068,UCZ1LcZESjYqzaQRhjdZJFwg,The fact they're holding each other back while...,Positive,0,0,2025-01-13 23:51:46,AU,17
2,UgxB0jh2Ur41mcXr5IB4AaABAg,papg2tsoFzg,Welcome to Javascript Course,@Abdulla-ip8qr,UCWBK35w5Swy1iF5xIbEyw3A,waiting next video will be?,Neutral,1,0,2020-07-06 13:18:16,IN,27
3,UgwMOh95MfK0GuXLLrF4AaABAg,31KTdfRH6nY,Building web applications in Java with Spring ...,@finnianthehuman,UCwQ2Z03nOcMxWozBb_Cv66w,Thanks for the great video.\n\nI don't underst...,Neutral,0,1,2024-09-18 12:04:12,US,27
4,UgxJuUe5ysG8OSbABAl4AaABAg,-hV6aeyPHPA,After a new engine her car dies on her way hom...,@ryoutubeplaylistb6137,UCTTcJ0tsAKQokmHB2qVb1qQ,Good person helping good people.\nThis is how ...,Positive,3,1,2025-01-10 19:39:03,US,2



⏱️ Cell executed in: 0.011 sec


In [16]:
df=df[['CommentText','Sentiment']]
df['Sentiment']=df['Sentiment'].map({"Positive":2,"Negative":0, 'Neutral':1})


⏱️ Cell executed in: 0.067 sec


In [17]:

df=df.dropna()


⏱️ Cell executed in: 0.065 sec


In [18]:
df = df.rename(columns={"CommentText": "text"})
df = df.rename(columns={"Sentiment": "label"})

df = df[["text", "label"]]


⏱️ Cell executed in: 0.086 sec


In [19]:
df.head()

,text,label
0,Anyone know what movie this is?,1
1,The fact they're holding each other back while...,2
2,waiting next video will be?,1
3,Thanks for the great video.\n\nI don't underst...,1
4,Good person helping good people.\nThis is how ...,2



⏱️ Cell executed in: 0.004 sec


In [20]:
print(df["label"].unique())

[1 2 0]

⏱️ Cell executed in: 0.006 sec


In [1]:
import pandas as pd
import re

# always reload fresh data so previous in-memory sampling doesn't leak in
df = pd.read_csv(r"D:\ORBIT\COMMENT_ANALYZER\datasets\Sentiment.csv")
df = df[["CommentText", "Sentiment"]]
df["Sentiment"] = df["Sentiment"].map({"Positive": 2, "Negative": 0, "Neutral": 1})
df = df.rename(columns={"CommentText": "text", "Sentiment": "label"})
df = df[["text", "label"]]


def clean_text(text):
    text = str(text)

    # remove newline characters
    text = text.replace("\n", " ")

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # remove non-ascii
    text = re.sub(r"[^\x00-\x7F]+", "", text)

    return text.strip()

# apply cleaning
df["text"] = df["text"].apply(clean_text)

# remove nulls and tiny text
df = df.dropna()
df = df[df["text"].str.len() > 5]

# dataset-size control (faster than full 1M, but much larger than 50k)
USE_SUBSET = True            # True = sample subset for faster runs
MAX_ROWS = 150_000           # train on 1.5 lakh rows

if USE_SUBSET:
    sample_n = min(MAX_ROWS, len(df))
    df = df.sample(sample_n, random_state=42)

print("Final dataset size:", len(df))
print(df.head())

Final dataset size: 150000
                                                     text  label
46508   The fact that hes wiser and smarter than most ...      2
619633  He as an adult, was able to walk away from his...      2
20690   There's nothing stronger than fighting for you...      2
577273  I bet the insurance companies are chomping at ...      0
668551  Am so glad the other young lady helped her tha...      2

⏱️ Cell executed in: 15.742 sec


In [2]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, 
    test_size=0.1, 
    stratify=df["label"], 
    random_state=42
)


⏱️ Cell executed in: 5.506 sec


In [3]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

d:\python_envs\torch\torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\python_envs\torch\torch_gpu\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



⏱️ Cell executed in: 7.413 sec


In [24]:
sample = train_df["text"].iloc[0]

encoded = tokenizer(
    sample,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded)

{'input_ids': tensor([[  101,  2012,  2054,  2754,  2079,  2027,  2131,  2000,  1996,  2391,
          2073, 12553,  4853,  6165,  2003,  2061,  2204,  2008,  2027,  2064,
          5271,  2896, 28699,  5329,  2012,  1037,  3020,  3976,  2007,  1996,
          1042,  4523,  4481, 13359,  2111,  2049,  1037,  2488,  4003,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [4]:
import torch
from torch.utils.data import Dataset

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


⏱️ Cell executed in: 0.015 sec


In [5]:
train_dataset = CommentDataset(
    train_df["text"],
    train_df["label"],
    tokenizer
)

val_dataset = CommentDataset(
    val_df["text"],
    val_df["label"],
    tokenizer
)


⏱️ Cell executed in: 0.006 sec


In [28]:
sample = train_dataset[0]

print(sample["input_ids"].shape)      # should be [128]
print(sample["attention_mask"].shape) # should be [128]
print(sample["labels"])               # tensor(0/1/2)

torch.Size([128])
torch.Size([128])
tensor(0)

⏱️ Cell executed in: 0.003 sec


In [6]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3  # negative, neutral, positive
)

d:\python_envs\torch\torch_gpu\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



⏱️ Cell executed in: 1.542 sec


In [7]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

cuda

⏱️ Cell executed in: 0.171 sec


In [8]:
from torch.utils.data import DataLoader
import os

BATCH_SIZE = 16
# Windows + notebooks can hang with multiprocessing workers; keep 0 for stability.
NUM_WORKERS = 0 if os.name == "nt" else 2
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=False,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=False,
    drop_last=False,
)

print(
    f"DataLoader config -> batch_size={BATCH_SIZE}, "
    f"num_workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}, os={os.name}"
)

DataLoader config -> batch_size=16, num_workers=0, pin_memory=True, os=nt

⏱️ Cell executed in: 0.000 sec


In [32]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

print(outputs.loss)
print(outputs.logits.shape)

tensor(1.1131, device='cuda:0', grad_fn=<NllLossBackward0>)
torch.Size([16, 3])

⏱️ Cell executed in: 1.263 sec


In [33]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)


⏱️ Cell executed in: 0.014 sec


d:\python_envs\torch\torch_gpu\lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [9]:
from tqdm import tqdm

def train_epoch(model, data_loader, optimizer, device):
    model.train()
    
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(data_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    return total_loss / len(data_loader), acc


⏱️ Cell executed in: 0.001 sec


In [10]:
def eval_model(model, data_loader, device):
    model.eval()
    
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(data_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    return total_loss / len(data_loader), acc


⏱️ Cell executed in: 0.002 sec


In [36]:
EPOCHS = 1

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = eval_model(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")


Epoch 1/1


  7%|▋         | 776/11250 [04:34<1:01:50,  2.82it/s]


KeyboardInterrupt: 


⏱️ Cell executed in: 274.998 sec


In [37]:
from torch.utils.data import DataLoader, Subset
import torch
import copy

# quick small-dataset smoke check
small_train_n = min(512, len(train_dataset))
small_val_n = min(512, len(val_dataset))

g = torch.Generator().manual_seed(42)
train_idx = torch.randperm(len(train_dataset), generator=g)[:small_train_n].tolist()
val_idx = torch.randperm(len(val_dataset), generator=g)[:small_val_n].tolist()

small_train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=32, shuffle=True)
small_val_loader = DataLoader(Subset(val_dataset, val_idx), batch_size=32)

smoke_model = copy.deepcopy(model).to(device)
smoke_optimizer = AdamW(smoke_model.parameters(), lr=5e-5)

print(f"Small split sizes => train: {small_train_n}, val: {small_val_n}")

base_val_loss, base_val_acc = eval_model(smoke_model, small_val_loader, device)
print(f"Before quick training => Val Loss: {base_val_loss:.4f} | Val Acc: {base_val_acc:.4f}")

train_loss, train_acc = train_epoch(smoke_model, small_train_loader, smoke_optimizer, device)
val_loss, val_acc = eval_model(smoke_model, small_val_loader, device)

print(f"After 1 small epoch => Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
print(f"After 1 small epoch => Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

d:\python_envs\torch\torch_gpu\lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Small split sizes => train: 512, val: 512


100%|██████████| 16/16 [00:04<00:00,  4.00it/s]


Before quick training => Val Loss: 0.7382 | Val Acc: 0.6836


100%|██████████| 16/16 [00:03<00:00,  4.14it/s]

After 1 small epoch => Train Loss: 0.8462 | Train Acc: 0.5938
After 1 small epoch => Val Loss:   0.7587 | Val Acc:   0.6660

⏱️ Cell executed in: 62.661 sec


In [38]:
# class balance and simple baseline context
label_counts = df['label'].value_counts().sort_index()
majority_class = label_counts.idxmax()
majority_acc = label_counts.max() / label_counts.sum()

print('Label counts (0=neg,1=neutral,2=pos):')
print(label_counts.to_dict())
print(f'Majority class: {majority_class}')
print(f'Majority-class baseline accuracy: {majority_acc:.4f}')

Label counts (0=neg,1=neutral,2=pos):
{0: 68298, 1: 65211, 2: 66491}
Majority class: 0
Majority-class baseline accuracy: 0.3415

⏱️ Cell executed in: 0.004 sec


In [11]:
from pathlib import Path
import torch
from torch.optim import AdamW as TorchAdamW
from transformers import DistilBertForSequenceClassification


def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=3,
    patience=2,
    checkpoint_path=r"D:\ORBIT\COMMENT_ANALYZER\Positive_Negative_Analyzer\results\best_distilbert_sentiment.pt",
):
    best_val_loss = float("inf")
    best_val_acc = 0.0
    epochs_no_improve = 0
    history = []

    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
        val_loss, val_acc = eval_model(model, val_loader, device)

        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_val_acc = val_acc
            epochs_no_improve = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                checkpoint_path,
            )
            print(f"✅ Saved new best model -> {checkpoint_path}")
        else:
            epochs_no_improve += 1
            print(f"No improvement. Early-stop counter: {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print("🛑 Early stopping triggered.")
            break

    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    print(f"\nBest Val Loss: {ckpt['best_val_loss']:.4f} | Best Val Acc: {ckpt['best_val_acc']:.4f}")
    return history, model


# ===== RUN CONFIG =====
RUN_TRAINING = True        # starts training when this cell is run
USE_SMALL_DEBUG = False    # False=full dataset training
NUM_EPOCHS = 5
PATIENCE = NUM_EPOCHS + 1  # effectively disables early stop before all epochs
LR = 2e-5

run_train_loader = small_train_loader if USE_SMALL_DEBUG else train_loader
run_val_loader = small_val_loader if USE_SMALL_DEBUG else val_loader

if RUN_TRAINING:
    # start fresh model for clean training
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=3,
    ).to(device)

    optimizer = TorchAdamW(model.parameters(), lr=LR, weight_decay=0.01)

    history, model = train_with_early_stopping(
        model=model,
        train_loader=run_train_loader,
        val_loader=run_val_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=NUM_EPOCHS,
        patience=PATIENCE,
    )

    print("\nTraining complete. Best checkpoint restored into `model`.")
else:
    print("Config ready ✅ Set RUN_TRAINING=True and run this cell to train with early stopping.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1/5


100%|██████████| 938/938 [02:11<00:00,  7.11it/s]


Train Loss: 0.6900 | Train Acc: 0.7006
Val Loss:   0.6414 | Val Acc:   0.7247
✅ Saved new best model -> D:\ORBIT\COMMENT_ANALYZER\Positive_Negative_Analyzer\results\best_distilbert_sentiment.pt

Epoch 2/5


100%|██████████| 938/938 [02:10<00:00,  7.18it/s]


...

Train Loss: 0.3887 | Train Acc: 0.8449
Val Loss:   0.7080 | Val Acc:   0.7266
No improvement. Early-stop counter: 1/6

Epoch 4/5


100%|██████████| 938/938 [02:10<00:00,  7.17it/s]


Train Loss: 0.2477 | Train Acc: 0.9050
Val Loss:   0.8835 | Val Acc:   0.7148
No improvement. Early-stop counter: 2/6

Epoch 5/5


100%|██████████| 938/938 [02:10<00:00,  7.18it/s]
C:\Users\hp\AppData\Local\Temp\ipykernel_15308\2847993237.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch

Train Loss: 0.1645 | Train Acc: 0.9389
Val Loss:   1.1213 | Val Acc:   0.7124
No improvement. Early-stop counter: 3/6

Best Val Loss: 0.6400 | Best Val Acc: 0.7291

Training complete. Best checkpoint restored into `model`.

⏱️ Cell executed in: 15258.962 sec


In [12]:
from pathlib import Path
import torch

final_dir = Path(r"D:\ORBIT\COMMENT_ANALYZER\Positive_Negative_Analyzer\results\final_distilbert_sentiment")
final_dir.mkdir(parents=True, exist_ok=True)

# Save full Hugging Face model + tokenizer (recommended for reload/inference)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

# Also save a plain state_dict file
final_state_path = final_dir / "final_model_state_dict.pt"
torch.save(model.state_dict(), final_state_path)

print(f"✅ Final model folder saved: {final_dir}")
print(f"✅ Final state_dict saved: {final_state_path}")

✅ Final model folder saved: D:\ORBIT\COMMENT_ANALYZER\Positive_Negative_Analyzer\results\final_distilbert_sentiment
✅ Final state_dict saved: D:\ORBIT\COMMENT_ANALYZER\Positive_Negative_Analyzer\results\final_distilbert_sentiment\final_model_state_dict.pt

⏱️ Cell executed in: 1.826 sec


In [15]:
import torch

# label mapping used in your training (0=Negative, 1=Neutral, 2=Positive)
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

test_comments = [
    "I do not hate this video, but I wouldn't watch it again.",
    "The tutorial is not bad, but the audio is terrible.",
    "I am not unhappy with the result, but it's not great either.",
    "Not only was it slow, but it was also confusing and unhelpful.",
    "It's not perfect, but honestly it's pretty useful.",
    "I expected better, but this is not the worst thing online.",
    "I am not impressed, but I appreciate the effort.",
    "This is not good, but not completely useless either.",
    "I can't say it's amazing, but it's not disappointing.",
    "Not what I wanted, but it still solved my problem.",
]

model.eval()
with torch.no_grad():
    encoded = tokenizer(
        test_comments,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    outputs = model(**encoded)
    probs = torch.softmax(outputs.logits, dim=1)
    preds = torch.argmax(probs, dim=1).cpu().tolist()
    confs = torch.max(probs, dim=1).values.cpu().tolist()

print("\nModel predictions on tricky comments:\n")
for text, pred, conf in zip(test_comments, preds, confs):
    print(f"Comment: {text}")
    print(f"Prediction: {id2label[pred]} | Confidence: {conf:.4f}")
    print("-" * 90)


Model predictions on tricky comments:

Comment: I do not hate this video, but I wouldn't watch it again.
Prediction: Negative | Confidence: 0.8117
------------------------------------------------------------------------------------------
Comment: The tutorial is not bad, but the audio is terrible.
Prediction: Negative | Confidence: 0.9952
------------------------------------------------------------------------------------------
Comment: I am not unhappy with the result, but it's not great either.
Prediction: Negative | Confidence: 0.6020
------------------------------------------------------------------------------------------
Comment: Not only was it slow, but it was also confusing and unhelpful.
Prediction: Negative | Confidence: 0.8183
------------------------------------------------------------------------------------------
Comment: It's not perfect, but honestly it's pretty useful.
Prediction: Positive | Confidence: 0.6654
---------------------------------------------------------